# Korean and Uzbek in a real tokenizer

A Tashkent electronics shop runs a support bot in English, Korean and Uzbek (Latin and Cyrillic), and a message
in Uzbek costs about twice as much as in English. In this notebook you load three real tokenizers with Hugging Face
Tokenizers, count the same four messages, look at the pieces that are smaller than a letter, and fix a Korean
product name that is stored in two Unicode forms.

About 10 minutes. Run the cells from top to bottom. Everything runs on a free CPU runtime, with no account and no key.
The three `tokenizer.json` files come from the Hugging Face Hub, about 20 MB together. No model is downloaded.

## 1. Install the Tokenizers version the lesson used

Colab already has the `tokenizers` library. The next cell installs the exact version the lesson ran (0.23.3), so
your counts match the lesson's.

**Look for:** no error. pip may print a few lines about other packages; you can ignore them.

In [1]:
%pip install -q tokenizers==0.23.3

Note: you may need to restart the kernel to use updated packages.


## 2. The four messages and the three tokenizers

`MESSAGES` holds the same bot message in four forms. `Tokenizer.from_pretrained` downloads a model's
`tokenizer.json` from the Hub; the string is the model's id there. The cell prints how big each vocabulary is and
which kind of tokenizer it is.

**Look for:** GPT-2 has 50,257 tokens in its vocabulary, Qwen2.5 has 151,665 and XLM-RoBERTa 250,002. The first two
are `BPE` (merges, as in the last lesson), XLM-RoBERTa is `Unigram`. You may see a warning about `HF_TOKEN`; ignore it.

In [2]:
import json, unicodedata
from tokenizers import Tokenizer

MESSAGES = {
    "en": "Your order has been shipped and will arrive tomorrow.",
    "ko": "주문하신 상품이 발송되었으며 내일 도착합니다.",
    "uz_lat": "Buyurtmangiz joʻnatildi va ertaga yetib keladi.",
    "uz_cyr": "Буюртмангиз жўнатилди ва эртага етиб келади.",
}
NAMES = {"gpt2": "openai-community/gpt2", "qwen": "Qwen/Qwen2.5-0.5B", "xlmr": "FacebookAI/xlm-roberta-base"}

tokenizers = {}
for key, name in NAMES.items():
    tokenizers[key] = Tokenizer.from_pretrained(name)
    model_type = json.loads(tokenizers[key].to_str())["model"]["type"]
    print(f"{key:5} {name:28} vocabulary: {tokenizers[key].get_vocab_size():>7,}   type: {model_type}")

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

gpt2  openai-community/gpt2        vocabulary:  50,257   type: BPE


tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

qwen  Qwen/Qwen2.5-0.5B            vocabulary: 151,665   type: BPE


tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

xlmr  FacebookAI/xlm-roberta-base  vocabulary: 250,002   type: Unigram


## 3. Count tokens on GPT-2's tokenizer

This is the function from the lesson's code screen. `add_special_tokens=False` counts only your text.
`len(encoding.ids)` is the number of tokens. The cell also prints each message's length in characters and in
UTF-8 bytes.

**Look for:** `{'en': 10, 'ko': 55, 'uz_lat': 21, 'uz_cyr': 52}`. The Korean message has only 25 characters
against 53 in English, yet it takes 55 tokens against 10.

In [3]:
def count_tokens(tokenizer, messages):
    counts = {}
    for script, text in messages.items():
        encoding = tokenizer.encode(
            text, add_special_tokens=False)
        counts[script] = len(encoding.ids)
    return counts

gpt2 = tokenizers["gpt2"]
print("GPT-2 counts:", count_tokens(gpt2, MESSAGES))
for script, text in MESSAGES.items():
    print(f"{script:7} characters: {len(text):3}   utf8 bytes: {len(text.encode('utf-8')):3}")

GPT-2 counts: {'en': 10, 'ko': 55, 'uz_lat': 21, 'uz_cyr': 52}
en      characters:  53   utf8 bytes:  53
ko      characters:  25   utf8 bytes:  65
uz_lat  characters:  47   utf8 bytes:  48
uz_cyr  characters:  44   utf8 bytes:  82


## 4. What the hosted model really billed

The shop pays for a hosted model, and you cannot download its tokenizer. The lesson sent each message once to
`google/gemini-2.5-flash-lite` (through OpenRouter, on 2026-11-13) and kept the API's answer. The cell holds the
part of those recorded answers that matters, `usage.prompt_tokens`, and reads it the same way you would read a live
answer. No request is sent.

**Look for:** English 11, Korean 15, Uzbek Latin 17, Uzbek Cyrillic 20. Uzbek Cyrillic is 1.8 times English: the
owner's bill. The gap is smaller than on GPT-2, but it is there.

In [4]:
# Copied from the lesson's recordings (bill-en, bill-ko, bill-uz-lat, bill-uz-cyr), trimmed to what is used.
# Each request was one user message with max_tokens=1.
RECORDED = {
    "en":     {"model": "google/gemini-2.5-flash-lite", "content": MESSAGES["en"],     "usage": {"prompt_tokens": 11, "completion_tokens": 1}},
    "ko":     {"model": "google/gemini-2.5-flash-lite", "content": MESSAGES["ko"],     "usage": {"prompt_tokens": 15, "completion_tokens": 1}},
    "uz_lat": {"model": "google/gemini-2.5-flash-lite", "content": MESSAGES["uz_lat"], "usage": {"prompt_tokens": 17, "completion_tokens": 1}},
    "uz_cyr": {"model": "google/gemini-2.5-flash-lite", "content": MESSAGES["uz_cyr"], "usage": {"prompt_tokens": 20, "completion_tokens": 1}},
}

def billed_input_tokens(answer):
    return answer["usage"]["prompt_tokens"]

english = billed_input_tokens(RECORDED["en"])
for script, answer in RECORDED.items():
    n = billed_input_tokens(answer)
    print(f"{script:7} billed input tokens: {n:3}   times English: {n / english:.1f}")

en      billed input tokens:  11   times English: 1.0
ko      billed input tokens:  15   times English: 1.4
uz_lat  billed input tokens:  17   times English: 1.5
uz_cyr  billed input tokens:  20   times English: 1.8


## 5. Pieces smaller than a letter

GPT-2's tokenizer starts from the 256 possible bytes and prints each byte as a stand-in character. The helper
`bytes_of` turns a token back into the bytes it holds. A Cyrillic letter is 2 bytes in UTF-8, so a token with only
1 byte is half a letter. The cell prints the tokens of the first word, `Буюртмангиз` ("your order").

**Look for:** 15 tokens for 11 letters. Eight tokens hold a single byte (`half a letter`). Four letters, Б, ю, г
and з, were never merged, so each one costs two tokens that mean nothing alone.

In [5]:
def bytes_to_unicode():
    # the GPT-2 byte <-> printable character table (used by byte-level BPE)
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return dict(zip(bs, [chr(c) for c in cs]))

BYTE_OF = {c: b for b, c in bytes_to_unicode().items()}

def bytes_of(token):
    return bytes(BYTE_OF[c] for c in token)

word = "Буюртмангиз"
enc = gpt2.encode(word, add_special_tokens=False)
half = 0
for t in enc.tokens:
    raw = bytes_of(t)
    if len(raw) == 1:
        half += 1
    print(f"{t:4} bytes: {raw.hex(' '):6} {'half a letter' if len(raw) == 1 else raw.decode('utf-8')}")
print(f"{len(enc.tokens)} tokens for {len(word)} letters, {half} of them half a letter")

Ð    bytes: d0     half a letter
ĳ    bytes: 91     half a letter
Ñĥ   bytes: d1 83  у
Ñ    bytes: d1     half a letter
İ    bytes: 8e     half a letter
ÑĢ   bytes: d1 80  р
ÑĤ   bytes: d1 82  т
Ð¼   bytes: d0 bc  м
Ð°   bytes: d0 b0  а
Ð½   bytes: d0 bd  н
Ð    bytes: d0     half a letter
³    bytes: b3     half a letter
Ð¸   bytes: d0 b8  и
Ð    bytes: d0     half a letter
·    bytes: b7     half a letter
15 tokens for 11 letters, 8 of them half a letter


## 6. Korean: three tokens for one syllable

A Korean syllable is 3 bytes. `offsets` says which characters each token came from.

**Look for:** the first three tokens all have offsets `(0, 1)`: all three come from the first character. Decoding
those three gives `주`. Decoding only two gives `�`, a broken character.

In [6]:
enc = gpt2.encode(MESSAGES["ko"], add_special_tokens=False)
print("first 7 tokens: ", enc.tokens[:7])
print("their offsets:  ", enc.offsets[:7])
print("decode first 3: ", gpt2.decode(enc.ids[:3]))
print("decode first 2: ", gpt2.decode(enc.ids[:2]))

first 7 tokens:  ['ì', '£', '¼', 'ë', '¬', '¸', 'íķ']
their offsets:   [(0, 1), (0, 1), (0, 1), (1, 2), (1, 2), (1, 2), (2, 3)]
decode first 3:  주
decode first 2:  �


## 7. The same function on three tokenizers

Nothing changes but the tokenizer. The cell prints the counts and how many times the English count each message
is. Then it shows the pieces: XLM-RoBERTa prints its tokens as text (`▁` marks a space before the piece), and Qwen2.5's
tokens are decoded from their bytes with the same `bytes_of` helper.

**Look for:** XLM-RoBERTa returns `{'en': 11, 'ko': 11, 'uz_lat': 16, 'uz_cyr': 19}`: Korean costs as much as
English there. Qwen2.5 gives 16 for Korean. On both newer tokenizers, Uzbek Cyrillic is the most expensive
relative to English (2.6 times and 1.7 times). XLM-RoBERTa keeps whole Korean words such as `▁주문` and `▁상품`, but
still cuts `эртага` (tomorrow) into `▁эр`, `та`, `га`. Qwen2.5 makes about one token per Korean syllable.

In [7]:
all_counts = {key: count_tokens(tok, MESSAGES) for key, tok in tokenizers.items()}
print("XLM-RoBERTa counts:", all_counts["xlmr"])
print()
print(f"{'message':8}" + "".join(f"{key:>14}" for key in tokenizers))
for script in MESSAGES:
    row = ""
    for key in tokenizers:
        n = all_counts[key][script]
        row += f"{n:>8} {n / all_counts[key]['en']:4.1f}x"
    print(f"{script:8}{row}")
print()
# the pieces: XLM-RoBERTa prints text; Qwen2.5 is byte-level like GPT-2, so decode each token from its bytes
print("XLM-RoBERTa, Korean:  ", tokenizers["xlmr"].encode(MESSAGES["ko"], add_special_tokens=False).tokens)
print("XLM-RoBERTa, Cyrillic:", tokenizers["xlmr"].encode(MESSAGES["uz_cyr"], add_special_tokens=False).tokens)
qwen_ko = tokenizers["qwen"].encode(MESSAGES["ko"], add_special_tokens=False).tokens
print("Qwen2.5, Korean:      ", [bytes_of(t).decode("utf-8") for t in qwen_ko])

XLM-RoBERTa counts: {'en': 11, 'ko': 11, 'uz_lat': 16, 'uz_cyr': 19}

message           gpt2          qwen          xlmr
en            10  1.0x      10  1.0x      11  1.0x
ko            55  5.5x      16  1.6x      11  1.0x
uz_lat        21  2.1x      19  1.9x      16  1.5x
uz_cyr        52  5.2x      26  2.6x      19  1.7x

XLM-RoBERTa, Korean:   ['▁주문', '하신', '▁상품', '이', '▁발송', '되었으며', '▁내', '일', '▁도착', '합니다', '.']
XLM-RoBERTa, Cyrillic: ['▁Бу', 'ю', 'рт', 'ман', 'гиз', '▁ж', 'ўна', 'тил', 'ди', '▁ва', '▁эр', 'та', 'га', '▁', 'ети', 'б', '▁ке', 'лади', '.']
Qwen2.5, Korean:       ['주', '문', '하신', ' 상', '품', '이', ' 발', '송', '되었', '으며', ' 내', '일', ' 도', '착', '합니다', '.']


## 8. What `add_special_tokens` adds

Left on (the default), some tokenizers add marker tokens around your text. They are billed too, so the lesson
counts with it off.

**Look for:** GPT-2 gives 10 both ways. XLM-RoBERTa gives 11 without and 13 with: it adds `<s>` at the start and
`</s>` at the end.

In [8]:
for key in ["gpt2", "xlmr"]:
    tok = tokenizers[key]
    without = tok.encode(MESSAGES["en"], add_special_tokens=False)
    with_markers = tok.encode(MESSAGES["en"])
    print(f"{key:5} without: {len(without.ids)}   with: {len(with_markers.ids)}   first and last token: {with_markers.tokens[0]!r} {with_markers.tokens[-1]!r}")

gpt2  without: 10   with: 10   first and last token: 'Your' '.'
xlmr  without: 11   with: 13   first and last token: '<s>' '</s>'


## 9. Broken on purpose: the same name, three times the tokens

An agent types a Korean product name. The supplier's catalogue has the same name, but stored decomposed (NFD): each
syllable is split into its letter parts. Here the catalogue line is made decomposed on purpose, to reproduce the
fault. The hosted model billed 9 input tokens for the typed name and 28 for the catalogue one (recorded answers
bill-ko-nfc and bill-ko-nfd).

**Look for:** the two lines print the same, but `equal: False`. The lengths are 13 and 27, the UTF-8 bytes 33 and
75. GPT-2 counts 29 and 72, XLM-RoBERTa 6 and 28. Qwen2.5 counts 13 for both: its `tokenizer.json` has an `NFC`
normalizer, GPT-2's has none (`None`).

In [9]:
typed_by_agent = "무선 이어폰 충전 케이스"                      # wireless earphone charging case
from_catalogue = unicodedata.normalize("NFD", typed_by_agent)    # the same words, stored decomposed

print("typed_by_agent:", typed_by_agent)
print("from_catalogue:", from_catalogue)
print("equal:", typed_by_agent == from_catalogue)
print("len:", len(typed_by_agent), len(from_catalogue))
print("first word code points:", [f"U+{ord(c):04X}" for c in typed_by_agent[:2]], [f"U+{ord(c):04X}" for c in from_catalogue[:5]])
print("utf8 bytes:", len(typed_by_agent.encode()), len(from_catalogue.encode()))
print()
# recorded answers bill-ko-nfc and bill-ko-nfd, trimmed
RECORDED_NAME = {"typed": {"usage": {"prompt_tokens": 9}}, "catalogue": {"usage": {"prompt_tokens": 28}}}
print("hosted model billed:", billed_input_tokens(RECORDED_NAME["typed"]), "and", billed_input_tokens(RECORDED_NAME["catalogue"]))
for key, tok in tokenizers.items():
    norm = json.loads(tok.to_str())["normalizer"]
    a = len(tok.encode(typed_by_agent, add_special_tokens=False).ids)
    b = len(tok.encode(from_catalogue, add_special_tokens=False).ids)
    print(f"{key:5} normalizer: {str(norm['type'] if norm else None):12} typed: {a:3}   catalogue: {b:3}")

typed_by_agent: 무선 이어폰 충전 케이스
from_catalogue: 무선 이어폰 충전 케이스
equal: False
len: 13 27
first word code points: ['U+BB34', 'U+C120'] ['U+1106', 'U+116E', 'U+1109', 'U+1165', 'U+11AB']
utf8 bytes: 33 75

hosted model billed: 9 and 28
gpt2  normalizer: None         typed:  29   catalogue:  72
qwen  normalizer: NFC          typed:  13   catalogue:  13
xlmr  normalizer: Precompiled  typed:   6   catalogue:  28


## 10. The fix: normalize to NFC

`unicodedata.normalize("NFC", text)` joins the letter parts back into whole syllables. Do it once, where text enters
your system, before you count it, send it or store it.

**Look for:** `equal: True`, and the catalogue line is back to 29 tokens on GPT-2 and 6 on XLM-RoBERTa.

In [10]:
fixed = unicodedata.normalize("NFC", from_catalogue)
print("equal:", fixed == typed_by_agent)
for key, tok in tokenizers.items():
    print(f"{key:5} after NFC: {len(tok.encode(fixed, add_special_tokens=False).ids)}")

equal: True
gpt2  after NFC: 29
qwen  after NFC: 13
xlmr  after NFC: 6


## 11. A look-alike that NFC does not fix

Uzbek Latin writes `oʻ` with the turned comma U+02BB. A keyboard gives the plain apostrophe U+0027. The two words look
alike but are different strings.

**Look for:** `equal: False`, still `False` after NFC and after NFKC, and the code points `U+02BB` and `U+0027`.

In [11]:
catalogue = "joʻnatildi"      # U+02BB
typed = "jo'natildi"          # ASCII apostrophe
print("equal:", catalogue == typed)
print("equal after NFC: ", unicodedata.normalize("NFC", catalogue) == unicodedata.normalize("NFC", typed))
print("equal after NFKC:", unicodedata.normalize("NFKC", catalogue) == unicodedata.normalize("NFKC", typed))
print("code points:", f"U+{ord(catalogue[2]):04X}", f"U+{ord(typed[2]):04X}")

equal: False
equal after NFC:  False
equal after NFKC: False
code points: U+02BB U+0027


## Now change it

Three small experiments. In each cell, edit the line marked `# <- edit` and run the cell again.

**Experiment 1: a tokenizer that learned Cyrillic.** The cell counts the half-letter tokens in `Буюртмангиз` on
GPT-2: 8 of 15. Change the name to `"qwen"`. The count drops to 0, because Qwen2.5 learned a merge for every Cyrillic
letter in the word. The word still takes 7 tokens: Qwen2.5 has the letters, but not the whole word.

In [12]:
key = "gpt2"  # <- edit: try "qwen"
tokens = tokenizers[key].encode("Буюртмангиз", add_special_tokens=False).tokens
print(key, "tokens:", len(tokens), "  half a letter:", sum(len(bytes_of(t)) == 1 for t in tokens))

gpt2 tokens: 15   half a letter: 8


**Experiment 2: map the apostrophe forms to one.** With `False`, the two Uzbek words from step 11 stay different.
Change it to `True`: the cell replaces the plain apostrophe and the right single quote with the turned comma
U+02BB, and the two become equal. This is the fix for a search box that misses `joʻnatildi`.

In [13]:
map_apostrophes = False  # <- edit: try True

def clean(text):
    if map_apostrophes:
        for ch in ["'", "\u2019"]:
            text = text.replace(ch, "\u02bb")
    return text

print("equal:", clean(catalogue) == clean(typed))

equal: False


**Experiment 3: decomposed Uzbek.** Korean is not the only script that NFD changes. With `"NFC"`, GPT-2 counts 52
tokens for the Uzbek Cyrillic message (44 characters), as in step 3. Change it to `"NFD"`: the one `ў` in the message
splits into `у` plus a separate accent mark, so the message has 45 characters and GPT-2 counts 53. Every such letter
in a longer text adds a token. On Qwen2.5 it stays 26, because its normalizer turns the text back into NFC first.

In [14]:
form = "NFC"  # <- edit: try "NFD"
text = unicodedata.normalize(form, MESSAGES["uz_cyr"])
print(form, "characters:", len(text))
for key in ["gpt2", "qwen"]:
    print(f"{key:5} tokens: {len(tokenizers[key].encode(text, add_special_tokens=False).ids)}")

NFC characters: 44
gpt2  tokens: 52
qwen  tokens: 26


## What you should see

- GPT-2's tokenizer prints 10 tokens for the English message and 55 for the Korean one.
- The first three Korean tokens all have offsets `(0, 1)`, and decoding them together gives `주`.
- The same function with XLM-RoBERTa returns `{'en': 11, 'ko': 11, 'uz_lat': 16, 'uz_cyr': 19}`.
- The decomposed product name has length 27 against 13, and GPT-2 counts 72 tokens against 29.
- After `unicodedata.normalize("NFC", ...)` the count is back to 29.